In [2]:
import os, sys, json, shutil, traceback, subprocess

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib

In [5]:
# DIRs

WORK_NAME = "M01_ds003768"


DATA_ROOT = Path("/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768")
RESULTS_ROOT = DATA_ROOT.parent / "rsCVR_Results"
WORK_ROOT = RESULTS_ROOT / WORK_NAME

INDEX_DIR = WORK_ROOT / "index"
LOG_DIR = WORK_ROOT / "logs"
QC_DIR = WORK_ROOT / "qc"
RAPIDTIDE_DIR = WORK_ROOT / "rapidtide"
SPM_DIR = WORK_ROOT / "spm_preproc"
SUMMARY_DIR = WORK_ROOT / "summary"
MANIFEST_DIR = WORK_ROOT / "manifests"

for d in [RESULTS_ROOT, WORK_ROOT, INDEX_DIR, LOG_DIR, QC_DIR, RAPIDTIDE_DIR, SPM_DIR, SUMMARY_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT    :", DATA_ROOT)
print("RESULTS_ROOT :", RESULTS_ROOT)
print("WORK_ROOT    :", WORK_ROOT)

# Fix Env for Rapidtide
ENV_BIN = Path(sys.executable).parent
os.environ["PATH"] = f"{ENV_BIN}:{os.environ['PATH']}"

RAPIDTIDE_EXE = ENV_BIN / "rapidtide"

print("Python executable :", sys.executable)
print("ENV_BIN           :", ENV_BIN)
print("rapidtide exists  :", RAPIDTIDE_EXE.exists())
print("which rapidtide   :", shutil.which("rapidtide"))

DATA_ROOT    : /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768
RESULTS_ROOT : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results
WORK_ROOT    : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768
Python executable : /home/bami/anaconda3/envs/JHA-spmCVR/bin/python
ENV_BIN           : /home/bami/anaconda3/envs/JHA-spmCVR/bin
rapidtide exists  : True
which rapidtide   : /home/bami/anaconda3/envs/JHA-spmCVR/bin/rapidtide


In [6]:
# Load Manifest file
run_manifest_csv = MANIFEST_DIR / "run_manifest.csv"
df_runs = pd.read_csv(run_manifest_csv)

print("run_manifest shape:", df_runs.shape)
df_runs

run_manifest shape: (66, 18)


,subject,run,t1_nii,t1_json,bold_nii,bold_json,spm_subject_root,spm_anat_dir,spm_run_dir,rapidtide_subject_root,rapidtide_run_dir,qc_subject_root,qc_run_dir,spm_preproc_bold,spm_motion_tsv,spm_brainmask,rapidtide_output_root,qc_summary_json
0,sub-01,run-1,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
1,sub-01,run-2,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
2,sub-02,run-1,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
3,sub-02,run-2,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
4,sub-03,run-1,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...,/media/bami/JHA/rsCVR/data/OpenNeu

In [ ]:
# Testing

In [7]:
TEST_SUBJECT = "sub-01"
TEST_RUN = "run-1"

test_row = df_runs[(df_runs["subject"] == TEST_SUBJECT) & (df_runs["run"] == TEST_RUN)].copy()

assert len(test_row) == 1, f"Expected exactly 1 row, got {len(test_row)}"
test_row = test_row.iloc[0]

test_row

subject                                                              sub-01
run                                                                   run-1
t1_nii                    /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...
t1_json                   /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...
bold_nii                  /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...
bold_json                 /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/...
spm_subject_root          /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
spm_anat_dir              /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
spm_run_dir               /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
rapidtide_subject_root    /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
rapidtide_run_dir         /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
qc_subject_root           /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
qc_run_dir                /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Res...
spm_preproc_

In [9]:
t1_nii = Path(test_row["t1_nii"])
t1_json = Path(test_row["t1_json"])
bold_nii = Path(test_row["bold_nii"])
bold_json = Path(test_row["bold_json"])

spm_subject_root = Path(test_row["spm_subject_root"])
spm_anat_dir = Path(test_row["spm_anat_dir"])
spm_run_dir = Path(test_row["spm_run_dir"])

rapidtide_subject_root = Path(test_row["rapidtide_subject_root"])
rapidtide_run_dir = Path(test_row["rapidtide_run_dir"])

qc_subject_root = Path(test_row["qc_subject_root"])
qc_run_dir = Path(test_row["qc_run_dir"])

spm_preproc_bold = Path(test_row["spm_preproc_bold"])
spm_motion_tsv = Path(test_row["spm_motion_tsv"])
spm_brainmask = Path(test_row["spm_brainmask"])
rapidtide_output_root = Path(test_row["rapidtide_output_root"])
qc_summary_json = Path(test_row["qc_summary_json"])

print("T1 NIfTI      :", t1_nii)
print("BOLD NIfTI    :", bold_nii)
print("SPM run dir   :", spm_run_dir)
print("RT output root:", rapidtide_output_root)

T1 NIfTI      : /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/anat/sub-01_T1w.nii.gz
BOLD NIfTI    : /media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/func/sub-01_task-rest_run-1_bold.nii.gz
SPM run dir   : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1
RT output root: /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/rapidtide/sub-01/run-1/sub-01_run-1


In [10]:
# IN/OUT DIRs
path_checks = {
    "t1_nii_exists": t1_nii.exists(),
    "t1_json_exists": t1_json.exists(),
    "bold_nii_exists": bold_nii.exists(),
    "bold_json_exists": bold_json.exists(),
    "spm_subject_root_exists": spm_subject_root.exists(),
    "spm_anat_dir_exists": spm_anat_dir.exists(),
    "spm_run_dir_exists": spm_run_dir.exists(),
    "rapidtide_run_dir_exists": rapidtide_run_dir.exists(),
    "qc_run_dir_exists": qc_run_dir.exists(),
}

path_checks

{'t1_nii_exists': True,
 't1_json_exists': True,
 'bold_nii_exists': True,
 'bold_json_exists': True,
 'spm_subject_root_exists': True,
 'spm_anat_dir_exists': True,
 'spm_run_dir_exists': True,
 'rapidtide_run_dir_exists': True,
 'qc_run_dir_exists': True}

In [11]:
assert all(path_checks.values()), f"Some required paths are missing: {path_checks}"
print("All required input/output paths exist.")

All required input/output paths exist.


In [12]:
# JSON meta
def load_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r") as f:
        return json.load(f)

t1_meta = load_json_safe(t1_json)
bold_meta = load_json_safe(bold_json)

print("T1 keys (sample):", list(t1_meta.keys())[:10])
print("BOLD keys (sample):", list(bold_meta.keys())[:10])
print("BOLD TR:", bold_meta.get("RepetitionTime", None))
print("TaskName:", bold_meta.get("TaskName", None))

T1 keys (sample): ['Modality', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturersModelName', 'InstitutionName', 'InstitutionalDepartmentName', 'InstitutionAddress', 'DeviceSerialNumber', 'StationName', 'BodyPartExamined']
BOLD keys (sample): ['Modality', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturersModelName', 'InstitutionName', 'InstitutionalDepartmentName', 'InstitutionAddress', 'DeviceSerialNumber', 'StationName', 'BodyPartExamined']
BOLD TR: 2.1
TaskName: rest


In [13]:
# Image header/shape
def nifti_info(path):
    img = nib.load(str(path))
    return {
        "shape": img.shape,
        "zooms": img.header.get_zooms(),
        "dtype": str(img.get_data_dtype()),
    }

t1_info = nifti_info(t1_nii)
bold_info = nifti_info(bold_nii)

print("T1 info  :", t1_info)
print("BOLD info:", bold_info)

T1 info  : {'shape': (192, 256, 256), 'zooms': (np.float32(0.9999971), np.float32(1.0), np.float32(1.0)), 'dtype': 'int16'}
BOLD info: {'shape': (80, 80, 35, 286), 'zooms': (np.float32(3.0), np.float32(3.0), np.float32(4.0), np.float32(2.1)), 'dtype': 'int16'}


In [14]:
# Sanity
assert len(t1_info["shape"]) == 3, f"T1 should be 3D, got {t1_info['shape']}"
assert len(bold_info["shape"]) == 4, f"BOLD should be 4D, got {bold_info['shape']}"
assert bold_meta.get("RepetitionTime", None) == 2.1, f"Expected TR 2.1, got {bold_meta.get('RepetitionTime', None)}"

print("Sanity checks passed.")

Sanity checks passed.


In [17]:
# SPM import
spm_test = {
    "import_ok": False,
    "defaults_ok": False,
    "error": None,
}

try:
    from spm import *
    spm_test["import_ok"] = True

except Exception:
    spm_test["error"] = traceback.format_exc()

spm_test

{'import_ok': True, 'defaults_ok': False, 'error': None}

In [18]:
# Rapidtide
rt_test = {
    "exe_exists": RAPIDTIDE_EXE.exists(),
    "help_ok": False,
    "error": None,
}

try:
    out = subprocess.run(
        [str(RAPIDTIDE_EXE), "--help"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False
    )
    rt_test["help_ok"] = (out.returncode == 0)
    rt_test["returncode"] = out.returncode
    rt_test["stdout_head"] = (out.stdout[:400] if out.stdout else out.stderr[:400])
except Exception:
    rt_test["error"] = traceback.format_exc()

rt_test

{'exe_exists': True,
 'help_ok': True,
 'error': None,
 'returncode': 0,
 'stdout_head': 'usage: rapidtide [-h]\n                 [--denoising | --delaymapping | --CVR | --globalpreselect]\n                 [--venousrefine | --nirs] [--brainmask MASK[:VALSPEC]]\n                 [--graymattermask MASK[:VALSPEC]]\n                 [--whitemattermask MASK[:VALSPEC]] [--csfmask MASK[:VALSPEC]]\n                 [--datatstep TSTEP | --datafreq FREQ] [--noantialias]\n                 [--invert] ['}

In [19]:
# PreProcessing test

preproc_spec = {
    "subject": TEST_SUBJECT,
    "run": TEST_RUN,

    "inputs": {
        "t1_nii": str(t1_nii),
        "t1_json": str(t1_json),
        "bold_nii": str(bold_nii),
        "bold_json": str(bold_json),
        "TR": bold_meta.get("RepetitionTime", None),
    },

    "outputs": {
        "spm_subject_root": str(spm_subject_root),
        "spm_anat_dir": str(spm_anat_dir),
        "spm_run_dir": str(spm_run_dir),

        "spm_preproc_bold": str(spm_preproc_bold),
        "spm_motion_tsv": str(spm_motion_tsv),
        "spm_brainmask": str(spm_brainmask),

        "rapidtide_run_dir": str(rapidtide_run_dir),
        "rapidtide_output_root": str(rapidtide_output_root),

        "qc_run_dir": str(qc_run_dir),
        "qc_summary_json": str(qc_summary_json),
    },

    "status": {
        "path_checks": path_checks,
        "spm_test": spm_test,
        "rapidtide_test": rt_test,
    }
}

preproc_spec

{'subject': 'sub-01',
 'run': 'run-1',
 'inputs': {'t1_nii': '/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/anat/sub-01_T1w.nii.gz',
  't1_json': '/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/anat/sub-01_T1w.json',
  'bold_nii': '/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/func/sub-01_task-rest_run-1_bold.nii.gz',
  'bold_json': '/media/bami/JHA/rsCVR/data/OpenNeuro/ds003768/sub-01/func/sub-01_task-rest_run-1_bold.json',
  'TR': 2.1},
 'outputs': {'spm_subject_root': '/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01',
  'spm_anat_dir': '/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/anat',
  'spm_run_dir': '/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1',
  'spm_preproc_bold': '/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1/sub-01_run-1_preproc_bold.nii.gz',
  'spm_motion_tsv': '/media/bami/JHA/rsCVR/data/OpenN

In [20]:
# Save Prep specs :: as .json
test_spec_json = spm_run_dir / f"{TEST_SUBJECT}_{TEST_RUN}_spm_test_spec.json"
with open(test_spec_json, "w") as f:
    json.dump(preproc_spec, f, indent=2)

print(test_spec_json)

/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1/sub-01_run-1_spm_test_spec.json


In [22]:
# SPM-Wrapper
import spm

def run_spm_preproc_test_safe(spec: dict, dry_run: bool = True):
    result = {
        "dry_run": dry_run,
        "subject": spec["subject"],
        "run": spec["run"],
        "success": False,
        "planned_steps": [
            "validate input T1/BOLD paths",
            "validate output directories",
            "import spm module",
            "reserve standard output filenames",
            "return execution plan"
        ],
        "executed": None,
        "error": None,
    }

    try:
        assert Path(spec["inputs"]["t1_nii"]).exists(), f"Missing T1: {spec['inputs']['t1_nii']}"
        assert Path(spec["inputs"]["bold_nii"]).exists(), f"Missing BOLD: {spec['inputs']['bold_nii']}"

        assert Path(spec["outputs"]["spm_subject_root"]).exists()
        assert Path(spec["outputs"]["spm_anat_dir"]).exists()
        assert Path(spec["outputs"]["spm_run_dir"]).exists()
        assert Path(spec["outputs"]["rapidtide_run_dir"]).exists()
        assert Path(spec["outputs"]["qc_run_dir"]).exists()

        if dry_run:
            result["success"] = True
            result["executed"] = "path validation only"
            return result

        # import-only SPM smoke test
        _ = spm
        result["success"] = True
        result["executed"] = "import spm"
        return result

    except Exception:
        result["error"] = traceback.format_exc()
        return result

In [23]:
# Dry-Run testing
wrapper_result_dry = run_spm_preproc_test_safe(preproc_spec, dry_run=True)
wrapper_result_dry

{'dry_run': True,
 'subject': 'sub-01',
 'run': 'run-1',
 'success': True,
 'planned_steps': ['validate input T1/BOLD paths',
  'validate output directories',
  'import spm module',
  'reserve standard output filenames',
  'return execution plan'],
 'executed': 'path validation only',
 'error': None}

In [24]:
# Executable testing
wrapper_result_exec = run_spm_preproc_test_safe(preproc_spec, dry_run=False)
wrapper_result_exec

{'dry_run': False,
 'subject': 'sub-01',
 'run': 'run-1',
 'success': True,
 'planned_steps': ['validate input T1/BOLD paths',
  'validate output directories',
  'import spm module',
  'reserve standard output filenames',
  'return execution plan'],
 'executed': 'import spm',
 'error': None}

In [25]:
# Rapidtide CLI testing
rt_test = {
    "exe_exists": RAPIDTIDE_EXE.exists(),
    "help_ok": False,
    "returncode": None,
    "stdout_head": None,
    "error": None,
}

try:
    out = subprocess.run(
        [str(RAPIDTIDE_EXE), "--help"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False
    )
    rt_test["returncode"] = out.returncode
    rt_test["help_ok"] = (out.returncode == 0)
    rt_test["stdout_head"] = out.stdout[:500] if out.stdout else out.stderr[:500]

except Exception:
    rt_test["error"] = traceback.format_exc()

rt_test

{'exe_exists': True,
 'help_ok': True,
 'returncode': 0,
 'stdout_head': 'usage: rapidtide [-h]\n                 [--denoising | --delaymapping | --CVR | --globalpreselect]\n                 [--venousrefine | --nirs] [--brainmask MASK[:VALSPEC]]\n                 [--graymattermask MASK[:VALSPEC]]\n                 [--whitemattermask MASK[:VALSPEC]] [--csfmask MASK[:VALSPEC]]\n                 [--datatstep TSTEP | --datafreq FREQ] [--noantialias]\n                 [--invert] [--interptype {univariate,cubic,quadratic}]\n                 [--offsettime OFFSETTIME] [--autosync] [',
 'error': None}

In [26]:
# Wrapper Result save

wrapper_test_json = spm_run_dir / f"{TEST_SUBJECT}_{TEST_RUN}_spm_wrapper_test.json"
with open(wrapper_test_json, "w") as f:
    json.dump({
        "preproc_spec": preproc_spec,
        "wrapper_result_dry": wrapper_result_dry,
        "wrapper_result_exec": wrapper_result_exec,
        "rapidtide_test": rt_test,
    }, f, indent=2)

print(wrapper_test_json)

/media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1/sub-01_run-1_spm_wrapper_test.json


In [27]:
# SUMMARY

print("Test subject/run      :", TEST_SUBJECT, TEST_RUN)
print("T1 exists             :", t1_nii.exists())
print("BOLD exists           :", bold_nii.exists())
print("SPM import-only OK    :", wrapper_result_exec["success"])
print("rapidtide help OK     :", rt_test["help_ok"])
print("Dry-run OK            :", wrapper_result_dry["success"])
print("Test spec JSON        :", test_spec_json)
print("Wrapper test JSON     :", wrapper_test_json)

Test subject/run      : sub-01 run-1
T1 exists             : True
BOLD exists           : True
SPM import-only OK    : True
rapidtide help OK     : True
Dry-run OK            : True
Test spec JSON        : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1/sub-01_run-1_spm_test_spec.json
Wrapper test JSON     : /media/bami/JHA/rsCVR/data/OpenNeuro/rsCVR_Results/M01_ds003768/spm_preproc/sub-01/run-1/sub-01_run-1_spm_wrapper_test.json
